# 01 — `Decimal` : précision exacte pour la finance

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre pourquoi `float` n'est pas adapté à l'argent
- créer des `Decimal` correctement (depuis une chaîne, jamais depuis un float)
- maîtriser `quantize` et les modes d'arrondi (`ROUND_HALF_UP`, `ROUND_HALF_EVEN`)
- utiliser le contexte décimal (précision globale, traps)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- modèle objet complet (classes, héritage, surcharge)
- typage moderne, dataclasses, Pydantic v2
- décorateurs, context managers, modules avancés
- SQL, tests, packaging

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- NumPy, Pandas, Polars (notebooks suivants dans cette section)

## Plan

1. Le problème : `0.1 + 0.2` en float
2. Créer un `Decimal`
3. Opérations arithmétiques
4. `quantize` et modes d'arrondi
5. Le contexte décimal
6. Cas pratique : calcul de TVA
7. Synthèse
8. Exercices

---

## 1. Le problème : `0.1 + 0.2` en float

In [ ]:
0.1 + 0.2


In [ ]:
0.1 + 0.2 == 0.3


En comptabilité, un écart de `5.55e-17` euros est **inacceptable**. Les règles d'arrondi doivent être **déterministes** et conformes aux normes fiscales.

---

## 2. Créer un `Decimal`

Toujours passer une **chaîne** pour obtenir une valeur exacte.

In [ ]:
from decimal import Decimal

Decimal('0.1') + Decimal('0.2')


In [ ]:
Decimal('0.1') + Decimal('0.2') == Decimal('0.3')


### ❌ Piège : `Decimal(0.1)` depuis un float

In [ ]:
Decimal(0.1)  # imprécision du float propagée


**Règle : toujours `Decimal('...')` avec une chaîne.**

---

## 3. Opérations arithmétiques

In [ ]:
Decimal('10.50') + Decimal('2.30')


In [ ]:
Decimal('100') * Decimal('0.20')


In [ ]:
Decimal('1') / Decimal('3')


In [ ]:
Decimal('10.50') % Decimal('3')


---

## 4. `quantize` et modes d'arrondi

`quantize` arrondit à un nombre de décimales donné, avec un mode explicite.

In [ ]:
from decimal import ROUND_HALF_UP, ROUND_HALF_EVEN

prix = Decimal('19.995')
prix.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)


In [ ]:
prix.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)


- **`ROUND_HALF_UP`** : arrondi commercial classique (`.5` → vers le haut).
- **`ROUND_HALF_EVEN`** : arrondi « banker's rounding » (`.5` → vers le pair le plus proche). C'est le défaut de Python (`round()`), mais pas celui de la TVA !

---

## 5. Le contexte décimal

Le contexte global contrôle la précision et les alertes.

In [ ]:
import decimal

ctx = decimal.getcontext()
ctx.prec


In [ ]:
with decimal.localcontext() as ctx:
    ctx.prec = 4
    print(Decimal('1') / Decimal('7'))


---

## 6. Cas pratique : calcul de TVA

Calculer le TTC avec un arrondi commercial à 2 décimales.

In [ ]:
def ttc(prix_ht: Decimal, taux_tva: Decimal = Decimal('0.20')) -> Decimal:
    '''Calcule le prix TTC arrondi au centime (arrondi commercial).'''
    return (prix_ht * (1 + taux_tva)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)


In [ ]:
ttc(Decimal('19.99'))


In [ ]:
ttc(Decimal('100.00'))


In [ ]:
ttc(Decimal('9.995'))


---

## Synthèse

| API | Rôle |
|---|---|
| `Decimal('10.50')` | Création exacte |
| `.quantize(Decimal('0.01'), rounding=...)` | Arrondi contrôlé |
| `ROUND_HALF_UP` | Arrondi commercial |
| `ROUND_HALF_EVEN` | Banker's rounding |
| `localcontext()` | Précision locale |


### Règles à retenir

1. **Jamais `Decimal(float)`.** Toujours `Decimal('str')`.
2. **`ROUND_HALF_UP` pour l'argent**, pas le défaut Python.
3. **`quantize` pour fixer le nombre de décimales.** Ne laissez pas la précision dériver.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Facture simple *(facile)*

Écrire `total_facture(articles: list[tuple[str, str, str]]) -> Decimal` qui reçoit des tuples (nom, prix_ht_str, quantite_str), calcule le TTC par ligne (TVA 20 %, arrondi commercial au centime) et renvoie le total.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
from decimal import Decimal, ROUND_HALF_UP

def total_facture(articles: list[tuple[str, str, str]]) -> Decimal:
    total = Decimal('0')
    for nom, prix_str, qte_str in articles:
        ligne_ht = Decimal(prix_str) * Decimal(qte_str)
        ligne_ttc = (ligne_ht * Decimal('1.20')).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
        total += ligne_ttc
    return total

arts = [('Stylo', '1.99', '10'), ('Cahier', '3.50', '5')]
print(total_facture(arts))
```

</details>

### Exercice 2 — Comparaison float vs Decimal *(moyen)*

Calculer la somme de 10 000 valeurs `0.01` avec `float` et avec `Decimal`. Comparer à `100.00`.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
from decimal import Decimal

somme_float = sum(0.01 for _ in range(10_000))
somme_dec = sum(Decimal('0.01') for _ in range(10_000))
print(f'float: {somme_float} == 100 ? {somme_float == 100.0}')
print(f'Decimal: {somme_dec} == 100 ? {somme_dec == Decimal("100")}')
```

</details>

### Exercice 3 — Simulateur de prêt *(difficile)*

Écrire une fonction `mensualite(capital: str, taux_annuel: str, nb_mois: int) -> Decimal` qui calcule la mensualité d'un prêt avec la formule classique `M = C * t / (1 - (1+t)^-n)` où `t = taux_annuel / 12`. Tout en `Decimal`, arrondi au centime.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
from decimal import Decimal, ROUND_HALF_UP

def mensualite(capital: str, taux_annuel: str, nb_mois: int) -> Decimal:
    c = Decimal(capital)
    t = Decimal(taux_annuel) / 12
    m = c * t / (1 - (1 + t) ** (-nb_mois))
    return m.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

print(mensualite('200000', '0.035', 240))
```

</details>

---

## Ressources externes

### Documentation officielle
- [`decimal`](https://docs.python.org/3/library/decimal.html)

### PEPs de référence
- **PEP 327** — Decimal Data Type